In [5]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mlg-ulb/creditcardfraud")

print("Path to dataset files:", path)

d:\git\depi\DEBI-ONL4_AIS2_S2\.pyt\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


100%|██████████| 66.0M/66.0M [00:18<00:00, 3.68MB/s]

Extracting files...


Path to dataset files: C:\Users\NOUR SOFT\.cache\kagglehub\datasets\mlg-ulb\creditcardfraud\versions\3


In [4]:
pip install kagglehub


   ---------------------------------------- 0/5 [tqdm]
   ---------------------------------------- 0/5 [tqdm]
   ---------------------------------------- 0/5 [tqdm]
   ---------------------------------------- 0/5 [tqdm]
   ---------------------------------------- 0/5 [tqdm]
   -------- ------------------------------- 1/5 [pyyaml]
   -------- ------------------------------- 1/5 [pyyaml]
   -------- ------------------------------- 1/5 [pyyaml]
   -------- ------------------------------- 1/5 [pyyaml]
   ---------------- ----------------------- 2/5 [protobuf]
   ---------------- ----------------------- 2/5 [protobuf]
   ---------------- ----------------------- 2/5 [protobuf]
   ---------------- ----------------------- 2/5 [protobuf]
   ---------------- ----------------------- 2/5 [protobuf]
   ---------------- ----------------------- 2/5 [protobuf]
   ---------------- ----------------------- 2/5 [protobuf]
   ---------------- ----------------------- 2/5 [protobuf]
   ---------------- ----

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import rcParams
%matplotlib inline
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import BaggingClassifier
from sklearn.model_selection import GridSearchCV

In [6]:
data = pd.read_csv(path + "/creditcard.csv")

In [7]:
data.head()

,Time,V1,V2,V3,V4,V5,V6,V7,V8,V9,...,V21,V22,V23,V24,V25,V26,V27,V28,Amount,Class
0,0.0,-1.359807,-0.072781,2.536347,1.378155,-0.338321,0.462388,0.239599,0.098698,0.363787,...,-0.018307,0.277838,-0.110474,0.066928,0.128539,-0.189115,0.133558,-0.021053,149.62,0
1,0.0,1.191857,0.266151,0.166480,0.448154,0.060018,-0.082361,-0.078803,0.085102,-0.255425,...,-0.225775,-0.638672,0.101288,-0.339846,0.167170,0.125895,-0.008983,0.014724,2.69,0
2,1.0,-1.358354,-1.340163,1.773209,0.379780,-0.503198,1.800499,0.791461,0.247676,-1.514654,...,0.247998,0.771679,0.909412,-0.689281,-0.327642,-0.139097,-0.055353,-0.059752,378.66,0
3,1.0,-0.966272,-0.185226,1.792993,-0.863291,-0.010309,1.247203,0.237609,0.377436,-1.387024,...,-0.108300,0.005274,-0.190321,-1.175575,0.647376,-0.221929,0.062723,0.061458,123.50,0
4,2.0,-1.158233,0.877737,1.548718,0.403034,-0.407193,0.095921,0.592941,-0.270533,0.817739,...,-0.009431,0.798278,-0.137458,0.141267,-0.206010,0.502292,0.219422,0.215153,69.99,0


In [8]:
data = data.drop("Time", axis=1)

In [9]:
x = data.drop("Class", axis=1)
y = data["Class"]

In [10]:
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)

In [11]:
from sklearn.ensemble import AdaBoostClassifier, RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix , accuracy_score

In [13]:
hyper_param = {'criterion':['entropy'], 
               'max_depth':[4, 6], 
               'n_estimators':[10, 20, 50],
               'max_features':['sqrt'],
               }

tree_GS_param = GridSearchCV(RandomForestClassifier(), hyper_param, cv=5, scoring ='roc_auc', n_jobs=-1)
tree_training = tree_GS_param.fit(x_train, y_train)
print("Best Hyper Parameters:\n", tree_GS_param.best_params_)
print("Best Score:\n", tree_GS_param.best_score_)
print("Best Estimator:\n", tree_GS_param.best_estimator_)

Best Hyper Parameters:
 {'criterion': 'entropy', 'max_depth': 6, 'max_features': 'sqrt', 'n_estimators': 50}
Best Score:
 0.9795253222281965
Best Estimator:
 RandomForestClassifier(criterion='entropy', max_depth=6, n_estimators=50)


In [15]:
cm = confusion_matrix(y_test, tree_GS_param.predict(x_test))
print("Confusion Matrix:\n", cm)
print("Classification Report:\n", classification_report(y_test, tree_GS_param.predict(x_test)))
print("Accuracy Score:\n", accuracy_score(y_test, tree_GS_param.predict(x_test)))
print("sensetivity:\n", cm[1][1]/(cm[1][0]+cm[1][1]))
print("specificity:\n", cm[0][0]/(cm[0][0]+cm[0][1]))

Confusion Matrix:
 [[56861     3]
 [   25    73]]
Classification Report:
               precision    recall  f1-score   support

           0       1.00      1.00      1.00     56864
           1       0.96      0.74      0.84        98

    accuracy                           1.00     56962
   macro avg       0.98      0.87      0.92     56962
weighted avg       1.00      1.00      1.00     56962

Accuracy Score:
 0.9995084442259752
sensetivity:
 0.7448979591836735
specificity:
 0.9999472425436128


In [16]:
from sklearn.ensemble import AdaBoostClassifier, GradientBoostingClassifier, GradientBoostingRegressor
gbc_param = {"n_estimators":[100, 200], "learning_rate":[0.01, 0.1, 1],
             "max_depth":[3, 4], "subsample":[0.8, 1.0],
             "min_samples_split":[2, 5], "min_samples_leaf":[1, 2]}

gbc_GS_param = GridSearchCV(GradientBoostingClassifier(), gbc_param, cv=5, scoring ='roc_auc', n_jobs=-1)
gbc_training = gbc_GS_param.fit(x_train, y_train)
print("Best Hyper Parameters:\n", gbc_GS_param.best_params_)
print("Best Score:\n", gbc_GS_param.best_score_)
print("Best Estimator:\n", gbc_GS_param.best_estimator_)

KeyboardInterrupt: 